# Module 7.4: Intelligent Alerting

**Duration:** 30 minutes | **Level:** 2 (builds on M7.1-M7.3)

Reduce false positives by 80-90% through statistical anomaly detection, alert aggregation, and automated remediation.

## Overview

Traditional threshold-based alerts create alert fatigue:
- 20-50 daily notifications
- 90% false positive rate
- Alert desensitization
- Missed real incidents

**This module teaches:**
1. Statistical anomaly detection using Prophet
2. Alert aggregation and deduplication
3. PagerDuty integration for on-call rotation
4. Automated remediation with runbooks

## Setup and Imports

In [ ]:
# Import core functionality from module
import sys
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

from l2_m7_intelligent_alerting import (
    AnomalyDetector,
    AlertAggregator,
    PagerDutyIntegration,
    RunbookAutomation,
    Alert,
    load_example_data
)

from config import Config

print("✅ Imports successful")
print(f"Environment: {Config.ENVIRONMENT}")

# Expected: ✅ Imports successful

## Core Problem Statement

**The Alert Fatigue Problem:**

Traditional threshold-based alerting (`if latency > 2s, alert`) suffers from:
- **Noise:** Normal traffic spikes trigger false positives
- **Context blindness:** No awareness of daily/weekly patterns
- **Individual alerts:** Each metric alerts separately
- **Manual escalation:** Engineers manually route incidents

**Example scenario:**
- E-commerce site at 10 AM (peak shopping time)
- Latency spikes from 0.15s → 0.25s
- Threshold alert fires: "Latency > 0.20s"
- **Reality:** This is normal for peak hours
- **Result:** False positive, ignored alert

In [ ]:
# Demonstrate the problem with threshold-based alerting
def threshold_alert(value, threshold=0.20):
    """Traditional threshold-based alerting."""
    return value > threshold

# Simulate peak hour traffic
peak_hour_latencies = [0.15, 0.22, 0.25, 0.19, 0.23]  # Normal peak traffic
false_positives = sum(threshold_alert(v) for v in peak_hour_latencies)

print(f"Peak hour latencies: {peak_hour_latencies}")
print(f"False positive alerts: {false_positives}/{len(peak_hour_latencies)}")
print(f"False positive rate: {false_positives/len(peak_hour_latencies)*100:.0f}%")

# Expected: 60% false positive rate during normal peak hours

## Step 1: Anomaly Detection with Prophet

**Approach:** Use Facebook Prophet to learn time-series patterns and detect statistical deviations.

**Key concepts:**
- **Baseline training:** 7+ days of historical metrics
- **Seasonality:** Daily/weekly traffic patterns
- **Confidence bounds:** 99.7% interval (3-sigma)
- **Anomaly threshold:** Values outside predicted bounds

**Parameters:**
- `interval_width=0.997` → 99.7% confidence bounds
- `seasonality_mode='multiplicative'` → Traffic-pattern sensitive
- `std_threshold=3.0` → 3-sigma deviation threshold

In [ ]:
# Train anomaly detector on synthetic baseline data
print("Training anomaly detector...")

# Generate 7 days of synthetic baseline (1-minute resolution)
baseline_df = pd.DataFrame({
    'ds': pd.date_range(start='2025-11-01', periods=1000, freq='1min'),
    'y': np.random.normal(0.15, 0.02, 1000)  # Mean 0.15s, std 0.02s
})

# Initialize and train detector
detector = AnomalyDetector(std_threshold=3.0)
success = detector.train(baseline_df)

print(f"✅ Training complete: {len(baseline_df)} data points")
print(f"Model trained: {detector.trained}")

# Expected: Model trained successfully with 1000 data points

In [ ]:
# Test anomaly detection on different scenarios
test_scenarios = [
    {"name": "Normal operation", "value": 0.16, "expected_anomaly": False},
    {"name": "High latency spike", "value": 2.5, "expected_anomaly": True},
    {"name": "Moderate degradation", "value": 0.45, "expected_anomaly": True},
]

print("Testing anomaly detection:\\n")
for scenario in test_scenarios:
    result = detector.detect({
        'timestamp': datetime(2025, 11, 7, 10, 0),
        'value': scenario['value']
    })
    
    status = "🔴 ANOMALY" if result.is_anomaly else "🟢 NORMAL"
    print(f"{status} | {scenario['name']}: {scenario['value']:.2f}s")
    if result.is_anomaly:
        print(f"  → Severity: {result.severity}, σ={result.sigma_deviation:.2f}")

# Expected: Normal=🟢, Spikes=🔴

## Step 2: Alert Aggregation & Deduplication

**Problem:** Multiple related alerts flood notification channels.

**Example:**
- High latency detected
- Cache miss rate elevated
- Timeout errors increasing
- Connection pool warnings

**All symptoms of the same root cause!**

**Solution:** Group alerts by:
- **Service:** Same microservice/component
- **Time window:** Within 5-minute window
- **Result:** 10 alerts → 1 coherent incident

In [ ]:
# Demonstrate alert aggregation
aggregator = AlertAggregator(window_seconds=300)  # 5-minute window

# Simulate multiple related alerts
base_time = datetime(2025, 11, 7, 10, 5, 0)
alerts = [
    Alert(
        id="alert_1",
        metric="http_request_duration_seconds",
        service="api-gateway",
        timestamp=base_time,
        severity="high",
        message="High latency detected: 2.5s"
    ),
    Alert(
        id="alert_2",
        metric="cache_miss_rate",
        service="api-gateway",
        timestamp=base_time + timedelta(seconds=30),
        severity="medium",
        message="Cache miss rate elevated: 85%"
    ),
    Alert(
        id="alert_3",
        metric="timeout_errors",
        service="api-gateway",
        timestamp=base_time + timedelta(seconds=60),
        severity="high",
        message="Timeout errors increasing: 15/min"
    ),
]

# Add alerts to aggregator
for alert in alerts:
    aggregator.add_alert(alert)

print(f"Added {len(alerts)} individual alerts")

# Aggregate into incidents
incidents = aggregator.aggregate()

print(f"\\n✅ Aggregated into {len(incidents)} incident(s):\\n")
for incident in incidents:
    print(f"Incident: {incident.id}")
    print(f"  Service: {incident.service}")
    print(f"  Severity: {incident.severity}")
    print(f"  Alerts: {incident.alert_count}")
    print(f"  Summary: {incident.summary}")

# Expected: 3 alerts → 1 incident

## Step 3: On-Call Rotation with PagerDuty

**Purpose:** Automatically escalate incidents to on-call engineers.

**Features:**
- **Automated routing:** Incidents routed to current on-call
- **Escalation policies:** Automatic escalation if unacknowledged
- **Deduplication:** Similar incidents grouped automatically
- **Service mapping:** Route to appropriate team

**Integration via `pdpyras` Python client.**

In [ ]:
# PagerDuty integration (gracefully skips without credentials)
pd_integration = PagerDutyIntegration()

if Config.has_pagerduty():
    print("✅ PagerDuty configured")
else:
    print("⚠️  PagerDuty not configured (demo mode)")

# Attempt to create incident (will skip if no credentials)
if incidents:
    result = pd_integration.create_incident(incidents[0])
    
    if result:
        print(f"✅ PagerDuty incident created: {result.get('id')}")
    else:
        print("⚠️  Skipped PagerDuty incident creation (no credentials)")
else:
    print("No incidents to escalate")

# Expected: Skips gracefully without credentials

## Step 4: Runbook Automation (Auto-Remediation)

**Goal:** Automatically fix common issues without human intervention.

**Common runbooks:**
1. **Cache full** → Flush LRU entries (20%)
2. **Connection pool exhausted** → Graceful service restart
3. **Rate limits hit** → Enable aggressive caching

**Safety considerations:**
- Enable only for well-tested runbooks
- Track success/failure rates
- Human override capability
- Audit logging for all actions

In [ ]:
# Initialize runbook automation
automation = RunbookAutomation(enabled=True)

print("Available runbooks:")
for runbook in automation.runbooks.keys():
    print(f"  - {runbook}")

# Expected: Lists available runbooks

In [ ]:
# Execute runbook examples
runbook_tests = [
    ("cache_full", {"percentage": 20}),
    ("connection_pool_exhausted", {"graceful": True, "timeout": 30}),
    ("rate_limit_hit", {"cache_ttl": 300}),
]

print("Executing runbooks:\\n")
for trigger, params in runbook_tests:
    result = automation.execute(trigger, params)
    status = "✅" if result["executed"] else "❌"
    print(f"{status} {trigger}: {result.get('result', result.get('reason'))}")

# Expected: All runbooks execute successfully (simulated)

## Reality Checks & Trade-offs (TVH Framework v2.0)

### What This Approach Doesn't Do

❌ **Handle scenarios with <7 days historical data**
- Prophet needs stable baseline
- New services lack training data

❌ **Distinguish anomalies from planned events**
- Black Friday sales spike → false positive
- Seasonal traffic changes confuse model

❌ **Guarantee zero false negatives**
- Statistical models have inherent uncertainty
- Not suitable for life-critical systems

❌ **Work during rapid infrastructure changes**
- Frequent architecture changes invalidate patterns
- Early-stage startups with weekly pivots

### Trade-offs Accepted

**Complexity vs. Signal Clarity**
- More infrastructure (Prophet, PagerDuty)
- Longer setup time (7+ days baseline)
- **Benefit:** 80-90% false positive reduction

**Latency vs. Accuracy**
- Detection adds 200-500ms latency
- Not suitable for real-time streaming
- **Benefit:** Significantly better accuracy

**Cost vs. Alert Quality**
- Additional services (Prometheus, PagerDuty)
- ~$70-650/month operational cost
- **Benefit:** Prevents alert desensitization

## Common Implementation Failures

### Failure #1: Alert Fatigue Persists
**Symptoms:** Still getting 20+ alerts daily

**Root Cause:** Threshold too sensitive (σ < 2.5)

**Fix:**
```python
# Increase threshold
detector = AnomalyDetector(std_threshold=4.0)
```

### Failure #2: Missed Critical Incidents
**Symptoms:** Gradual degradation not detected

**Root Cause:** Over-tuned for false positives

**Fix:** Add hybrid approach with absolute thresholds

In [ ]:
# Demonstrate failure scenario: False positive from planned event
print("Scenario: Black Friday traffic spike\\n")

# Simulate Black Friday traffic (10x normal)
black_friday_value = 1.5  # 10x normal 0.15s baseline

result = detector.detect({
    'timestamp': datetime(2025, 11, 29, 10, 0),  # Black Friday
    'value': black_friday_value
})

if result.is_anomaly:
    print("🔴 FALSE POSITIVE: Legitimate traffic flagged as anomaly")
    print(f"   Severity: {result.severity}")
    print(f"\\n⚠️  FIX: Retrain model after planned high-traffic events")
else:
    print("🟢 No false positive (model adapted)")

# Expected: Likely false positive without event context

## Decision Card

| Scenario | Recommendation | Rationale |
|----------|----------------|-----------|
| **Mature system, 6+ months data** | ✅ Use intelligent alerting | Stable baseline available |
| **New service, <7 days data** | ⚠️ Simple thresholds first | Insufficient training data |
| **Low alert volume (<10/week)** | ⚠️ Skip aggregation | Overhead exceeds benefit |
| **Unpredictable traffic** | ⚠️ Hybrid model | Combine anomaly + manual thresholds |
| **Critical systems** | ⚠️ Human review gate | Zero false negatives required |
| **Rapid infrastructure changes** | ❌ Wait for stability | Historical patterns invalid |

### When NOT to Use

1. **<7 days operational data** - Insufficient baseline
2. **Low alert volume** - Aggregation complexity not worth it
3. **Unpredictable patterns** - Event-driven systems confuse models
4. **Zero false-negative tolerance** - Healthcare, financial systems
5. **Early-stage changes** - Daily architecture shifts

## Production Deployment Checklist

Before deploying to production:

- [ ] 7+ days of Prometheus baseline data collected
- [ ] Prophet model trained and validated on historical incidents
- [ ] PagerDuty integration tested with incident acknowledgment
- [ ] Auto-remediation runbooks tested in staging
- [ ] Alert aggregation rules tuned (<5% false positive rate)
- [ ] Escalation policies configured with backup on-call
- [ ] Monitoring dashboards showing detection accuracy
- [ ] Weekly alert quality reviews scheduled

### Scaling Considerations

**Data Volume:**
- Training: 7 days × 1-min resolution = ~10,080 points
- Manageable on standard hardware

**Latency:**
- Detection: ~200-500ms per check
- Evaluation window: 5-minute intervals

**Cost (Monthly):**
- Prometheus storage: $50-100
- PagerDuty: $0-500
- Compute: $20-50
- **Total: $70-650/month**

## Key Takeaways

### The Big Picture

**Intelligent alerting trades complexity for signal clarity.**

- **80-90% reduction** in false positives
- **Justifies overhead** for systems with >10 alerts/day
- **Requires stable baseline** (7+ days)
- **Not suitable** for new services or unpredictable traffic

### Implementation Path

1. **Deploy on non-critical metrics first** (2 weeks)
2. **Monitor false positive rates** (tune σ threshold)
3. **Gradually expand** to critical services
4. **Implement feedback mechanisms** (continuous improvement)

### Success Metrics

- False positive rate: <5% (down from 90%)
- Alert volume: 1-2 actionable incidents/day (down from 20-50)
- MTTR (Mean Time To Resolve): Reduced by 40-60%
- On-call engineer satisfaction: Significantly improved

---

**Next Module:** Module 8 - Advanced Observability Topics

**Further Learning:**
- Prophet documentation
- PagerDuty best practices
- SRE alerting strategies